In [ ]:
%pip install openai

In [1]:
import torch
from PIL import Image
import openai
from transformers import CLIPProcessor, CLIPModel
from openai import OpenAI


In [ ]:

# Load CLIP model and processor
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")


In [3]:

# OpenAI API key setup
client = OpenAI(api_key=OPENAI_API_KEY)
#openai.api_key = OPENAI_API_KEY

In [4]:

def extract_clip_features(image_path):
    # Load image
    image = Image.open(image_path).convert("RGB")
    
    # Process and generate features
    inputs = processor(images=image, return_tensors="pt")
    features = clip_model.get_image_features(**inputs)
    return features.squeeze().tolist()  # Convert to list for prompt use


In [72]:

def generate_description_from_features(features):
    # Summarize features for prompt
    feature_summary = ", ".join(f"{f:.2f}" for f in features[:20])  # Using the first 20 for simplicity
    
    # Construct prompt
    prompt = (
        f"Based on these image features: {feature_summary}\n\n"
        "Please provide a detailed description of the clothing item, including:\n"
        "- Product Name\n"
        "- Clothing Type\n"
        "- Main Colors\n"
        "- Texture and Material\n"
        "- Detailed Description"
    )
    messages=[{"role": "system", "content": "You are an AI assistant helping with feature extraction."},{"role": "user", "content": prompt}]
    # Use OpenAI's GPT API to generate description
    response = client.chat.completions.create(
        messages = messages,
        model="gpt-4o",  # Use "gpt-3.5-turbo" or "gpt-4" depending on your access
        max_tokens=150,
        temperature=0.7
    )
    
    # Extract and return generated description
    #description = response['choices'][0]['message']['content'].strip()
    return response


In [11]:
feature_summary = ", ".join(f"{f:.2f}" for f in features[:20])

In [12]:
prompt = (
        f"Based on these image features: {feature_summary}\n\n"
        "Please provide a detailed description of the clothing item, including:\n"
        "- Product Name\n"
        "- Clothing Type\n"
        "- Main Colors\n"
        "- Texture and Material\n"
        "- Detailed Description")

In [ ]:
prompt

In [6]:

# Example usage
image_path = r"C:\Users\pavan\Downloads\zalando-hd-resized\test\cloth\14673_00.jpg"  # Replace with your image path
features = extract_clip_features(image_path)


In [ ]:
len(features)

In [ ]:
import base64
import openai
from openai import OpenAI

OPENAI_API_KEY = 'KEY' # Replace with your OpenAI API key
client = OpenAI(api_key=OPENAI_API_KEY)
# Function to encode the image
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# Path to your image
image_path = "path_to_your_image.jpg"

# Getting the base64 string
base64_image = encode_image(image_path)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
    {
    "role": "user",
    "content": [
        {
        "type": "text",
        "text": "Generate features of the cloth",
        },
        {
        "type": "image_url",
        "image_url": {
        "url":  f"data:image/jpeg;base64,{base64_image}"
        },
        },
    ],
}
],
)

print(response.choices[0])

In [26]:
s = str(response.choices[0])

In [ ]:
response.choices[0]

In [27]:
import re

In [ ]:
if re.search('content',s):
    print(s)

In [33]:
match = re.search(r"content='(.*?)', refusal=None", s)

In [ ]:
match

In [ ]:
match.group(1)

In [ ]:
re.findall('content',s)

In [ ]:
import re

response_text = """Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The image depicts a gray sleeveless shirt featuring a triangular logo design. The logo includes the brand name "GUESS®" along with the text "U.S.A." and "WASHED JEANS." There is also a question mark and some numbers in the design.', refusal=None, role='assistant', audio=None, function_call=None, tool_calls=None))"""

# Regular expression to capture the content between "content='" and "', refusal=None"
match = re.search(r"content='(.*?)', refusal=None", response_text)

# Extract and print the content if found
if match:
    content = match.group(1)
    print(content)
else:
    print("No match found.")


In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

# Load the BLIP model and processor
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

# Load the image
image_path = r"C:\Users\pavan\Downloads\zalando-hd-resized\test\cloth\14673_00.jpg"
image = Image.open(image_path)

# Process the image and get a description
inputs = processor(images=image, return_tensors="pt")
output = model.generate(**inputs)
description = processor.decode(output[0], skip_special_tokens=True)
print("Generated Description:", description)


In [75]:
#response = generate_description_from_features(features)
#description = response['choices'][0]['message']['content'].strip()
#print("Generated Description:\n", description)

In [ ]:
type(response)

In [ ]:
response

In [ ]:
len(features)

In [9]:
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


In [10]:

# Load the model and tokenizer
model_name = "gpt2"  # or any other model compatible with causal language modeling
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


In [11]:

# Set padding token
tokenizer.pad_token = tokenizer.eos_token


In [25]:

# Function to generate an expanded summary of the image features
def detailed_feature_summary(flattened_features):
    # Simulate detailed features from specific parts of the array
    dominant_colors = [f"Color {i+1}: Intensity {flattened_features[i]:.2f}" for i in range(3)]
    texture_roughness = "rough" if np.mean(flattened_features[10:20]) > 0.5 else "smooth"
    material_type = "denim" if flattened_features[21] > 0.7 else "cotton"
    pattern_type = "striped" if flattened_features[22] > 0.5 else "solid"
    style_elements = "casual, everyday wear" if flattened_features[23] > 0.5 else "formal attire"

    return {
        "colors": dominant_colors,
        "texture": texture_roughness,
        "material": material_type,
        "pattern": pattern_type,
        "style": style_elements
    }


In [26]:

# Generate a descriptive prompt using the detailed features
def generate_detailed_description(flattened_features):
    # Extract detailed feature summary
    feature_summary = detailed_feature_summary(flattened_features)
    
    # Construct a highly descriptive prompt
    prompt = (
        "The following detailed features describe an item of clothing based on an image analysis:\n\n"
        f"- Dominant Colors: {', '.join(feature_summary['colors'])}\n"
        f"- Texture: {feature_summary['texture']} texture\n"
        f"- Material: Made of {feature_summary['material']}\n"
        f"- Pattern: {feature_summary['pattern']} pattern\n"
        f"- Style: Suitable for {feature_summary['style']}\n\n"
        "Please generate a detailed description of this item, including:\n"
        "- An appropriate product name\n"
        "- Specific clothing type (e.g., 'jacket', 'dress')\n"
        "- Primary colors and shades\n"
        "- Detailed information on texture and material type\n"
        "- Suggested occasions or use cases\n"
        "- Any additional features that make this item unique"
    )

    # Tokenize with truncation and padding
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )

    # Generate the text description
    outputs = model.generate(
        inputs.input_ids,
        max_length=200,  # Increase if you need a more verbose description
        temperature=0.7,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode and return the generated description
    description = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return description


In [32]:

# Example flattened features array (replace with actual extracted features)
features = np.random.rand(2048)


In [ ]:
len(features)

In [ ]:

# Generate and print the detailed description
description = generate_detailed_description(features)
print("Generated Description:\n", description)


In [ ]:
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Load the model and tokenizer
model_name = "gpt2"  # Replace with the model you're using
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set padding token to eos_token to avoid the padding warning
tokenizer.pad_token = tokenizer.eos_token

# Function to generate an expanded summary of the image features
def detailed_feature_summary(flattened_features):
    # Simulate detailed features from specific parts of the array
    dominant_colors = [f"Color {i+1}: Intensity {flattened_features[i]:.2f}" for i in range(3)]
    texture_roughness = "rough" if np.mean(flattened_features[10:20]) > 0.5 else "smooth"
    material_type = "denim" if flattened_features[21] > 0.7 else "cotton"
    pattern_type = "striped" if flattened_features[22] > 0.5 else "solid"
    style_elements = "casual, everyday wear" if flattened_features[23] > 0.5 else "formal attire"

    return {
        "colors": dominant_colors,
        "texture": texture_roughness,
        "material": material_type,
        "pattern": pattern_type,
        "style": style_elements
    }

# Generate a descriptive prompt using the detailed features
def generate_detailed_description(flattened_features):
    # Extract detailed feature summary
    feature_summary = detailed_feature_summary(flattened_features)
    
    # Construct a highly descriptive prompt
    prompt = (
        "The following detailed features describe an item of clothing based on an image analysis:\n\n"
        f"- Dominant Colors: {', '.join(feature_summary['colors'])}\n"
        f"- Texture: {feature_summary['texture']} texture\n"
        f"- Material: Made of {feature_summary['material']}\n"
        f"- Pattern: {feature_summary['pattern']} pattern\n"
        f"- Style: Suitable for {feature_summary['style']}\n\n"
        "Please generate a detailed description of this item, including:\n"
        "- An appropriate product name\n"
        "- Specific clothing type (e.g., 'jacket', 'dress')\n"
        "- Primary colors and shades\n"
        "- Detailed information on texture and material type\n"
        "- Suggested occasions or use cases\n"
        "- Any additional features that make this item unique"
    )

    # Tokenize with truncation, padding, and attention mask
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )
    attention_mask = inputs['attention_mask']  # Explicitly set attention mask

    # Generate text with the attention mask
    outputs = model.generate(
        inputs.input_ids,
        attention_mask=attention_mask,
        max_length=200,  # Increase if you need a more verbose description
        temperature=0.7,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode and return the generated description
    description = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return description

# Example flattened features array (replace with actual extracted features)
flattened_features = np.random.rand(2048)

# Generate and print the detailed description
description = generate_detailed_description(flattened_features)
print("Generated Description:\n", description)


In [7]:
from transformers import pipeline


In [ ]:

# Load a text-generation pipeline
generator = pipeline("text-generation", model="gpt2")  # or another model, like T5 or BLOOM


In [15]:

def generate_description_with_huggingface(flattened_features):
    # Use the first few features for simplicity
    feature_summary = ", ".join([f"{val:.2f}" for val in flattened_features[:20]])

    # Create a prompt
    prompt = (
        f"Given these features of a clothing item: {feature_summary}\n\n"
        "Generate a detailed product description including:\n"
        "- Product Name\n"
        "- Clothing Type\n"
        "- Main Colors and Shade\n"
        "- Texture and Material Type\n"
        "- Detailed Description"
    )

    # Generate text
    result = generator(prompt, max_length=150, num_return_sequences=1)
    description = result[0]["generated_text"]
    return description


In [22]:
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


In [ ]:

# Load the model and tokenizer
model_name = "gpt2"  # Replace with the model you're using
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


In [24]:

# Set padding token
tokenizer.pad_token = tokenizer.eos_token


In [25]:

# Function to extract a summary of image features
def summarize_features(flattened_features):
    # Summarize by getting averages or key segments
    # For simplicity, we'll create a dummy summary based on random features
    avg_color = np.mean(flattened_features[:10])  # Just an example
    texture_roughness = np.mean(flattened_features[10:20])
    
    return {
        "dominant_color": f"Color intensity around {avg_color:.2f}",
        "texture": "Rough" if texture_roughness > 0.5 else "Smooth",
        "pattern": "Striped" if np.mean(flattened_features[20:30]) > 0.5 else "Plain"
    }


In [26]:

# Generate description from features
def generate_description(flattened_features):
    # Generate a summary of the features
    feature_summary = summarize_features(flattened_features)
    
    # Construct the prompt with the feature summary
    prompt = (
        f"The clothing item has the following characteristics:\n"
        f"- Dominant Color: {feature_summary['dominant_color']}\n"
        f"- Texture: {feature_summary['texture']}\n"
        f"- Pattern: {feature_summary['pattern']}\n\n"
        "Based on these characteristics, provide a detailed description including:\n"
        "- Product Name\n"
        "- Clothing Type\n"
        "- Main Colors and Shade\n"
        "- Texture and Material Type\n"
        "- Detailed Description"
    )

    # Tokenize with truncation and padding
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )

    # Generate text
    outputs = model.generate(
        inputs.input_ids,
        max_length=150,
        temperature=0.7,
        top_p=0.95,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode the output
    description = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return description


In [27]:

# Example flattened features
features = np.random.rand(2048)  # Replace with actual features


In [ ]:

# Generate and print the description
description = generate_description(features)
print("Generated Description:\n", description)
